# **Imports and Initialization**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
os.makedirs("/content/drive/MyDrive/SISSA_Thesis/config", exist_ok=True)

In [ ]:
!pip install google-genai

from google import genai
from google.colab import userdata

In [ ]:
GEMINI_API_KEY = userdata.get('gemini_api_key')
client = genai.Client(api_key=GEMINI_API_KEY)

# Sanity check
response = client.models.generate_content(
    model='gemini-3.1-flash-lite',
    contents='say: online'
)
print(response.text)

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 500, model: gemini-3.1-flash-lite\nPlease retry in 24.513167327s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-3.1-fl

# **Tunable Parameters**

In [ ]:
%%writefile /content/drive/MyDrive/SISSA_Thesis/config/pipeline_config.yaml

# ── Paths ────────────────────────────────────────────────────────────────────
paths:
  base:        "/content/drive/MyDrive/SISSA_Thesis"
  blocks_dir:  "/content/drive/MyDrive/SISSA_Thesis/blocks_parsed"   # Stage 1 output
  pairs_dir:   "/content/drive/MyDrive/SISSA_Thesis/pairs"    # Stage 2 output
  final_dir:   "/content/drive/MyDrive/SISSA_Thesis/final"    # Stage 3 output

# ── Gemini ───────────────────────────────────────────────────────────────────
gemini:
  model:            "gemini-3.1-flash-lite"   # free tier model
  api_key:          "gemini_api_key"          # replace before running
  max_retries:      4
  base_backoff_sec: 5                         # exponential backoff starting point
  max_tokens:       1024
  temperature:      0.4                       # low = more factual, less creative
  sleep_between_calls_sec: 8


# ── Instruction generation ───────────────────────────────────────────────────
generation:
  max_pairs_per_block: 2      # how many (instruction, output) pairs to generate per block
  min_output_chars:    80     # discard Gemini outputs shorter than this (likely malformed)
  max_total_chars:     3000   # Stage 3 length filter: instruction + input + output combined

# ── Dataset split ────────────────────────────────────────────────────────────
dataset:
  train_ratio:  0.90
  random_seed:  42

# ── Logging ──────────────────────────────────────────────────────────────────
logging:
  skipped_log: "/content/drive/MyDrive/SISSA_Thesis/data/pairs/skipped_blocks.log"


Overwriting /content/drive/MyDrive/SISSA_Thesis/config/pipeline_config.yaml
